# Hierarchical Clustering - Complete Implementation
## Theory + Practical Implementation

This notebook explains Hierarchical (Agglomerative) Clustering from scratch and demonstrates it on real-world datasets.

## Objectives
- Understand Hierarchical Clustering
- Visualize dendrograms
- Compare linkage methods
- Find optimal clusters
- Apply to a real dataset
- Evaluate clustering quality


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster


## Load a Real Dataset (Wine Dataset)

In [ ]:
wine=load_wine()
X=wine.data
feature_names=wine.feature_names

scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

print(X.shape)
pd.DataFrame(X,columns=feature_names).head()


## Build Dendrogram

In [ ]:
Z=linkage(X_scaled,method='ward')

plt.figure(figsize=(12,6))
dendrogram(Z,truncate_mode='lastp',p=25)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Clusters")
plt.ylabel("Distance")
plt.show()


## Agglomerative Clustering

In [ ]:
model=AgglomerativeClustering(n_clusters=3,linkage='ward')
labels=model.fit_predict(X_scaled)

print("Silhouette Score:",silhouette_score(X_scaled,labels))
pd.Series(labels).value_counts().sort_index()


## Compare Linkage Methods

In [ ]:
methods=["ward","complete","average","single"]
scores={}
for m in methods:
    model=AgglomerativeClustering(n_clusters=3,linkage=m)
    lab=model.fit_predict(X_scaled)
    scores[m]=silhouette_score(X_scaled,lab)
scores


In [ ]:
plt.figure(figsize=(6,4))
plt.bar(scores.keys(),scores.values())
plt.title("Silhouette Score by Linkage")
plt.show()


## Choosing Number of Clusters

In [ ]:
ks=range(2,8)
vals=[]
for k in ks:
    lab=AgglomerativeClustering(n_clusters=k,linkage='ward').fit_predict(X_scaled)
    vals.append(silhouette_score(X_scaled,lab))

plt.plot(list(ks),vals,marker='o')
plt.xlabel("Clusters")
plt.ylabel("Silhouette")
plt.grid(True)
plt.show()

print("Best K:",list(ks)[int(np.argmax(vals))])


## Comparison with K-Means on Non-Spherical Data

In [ ]:
X,y=make_moons(n_samples=300,noise=0.08,random_state=42)

km=KMeans(n_clusters=2,n_init=10,random_state=42)
km_labels=km.fit_predict(X)

hc=AgglomerativeClustering(n_clusters=2,linkage="single")
hc_labels=hc.fit_predict(X)

fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].scatter(X[:,0],X[:,1],c=km_labels)
ax[0].set_title("KMeans")
ax[1].scatter(X[:,0],X[:,1],c=hc_labels)
ax[1].set_title("Hierarchical (Single)")
plt.show()


# Conclusion

- Hierarchical clustering builds a tree of relationships.
- Dendrogram helps determine cluster structure.
- Ward linkage generally performs best on compact clusters.
- Single linkage works well for irregular cluster shapes.
- Silhouette score helps evaluate clustering quality.
